# Assignment 1 — Training pipeline

**Goal:** predict the annual medical cost (`charges`, in $) of an insurance
member, so the insurer can budget for claims and target prevention programs
(see REPORT.md for the business framing).

Steps in this notebook:

1. Load the data
2. Clean it
3. Engineer features
4. Split into train / validation / test
5. Standardize the features

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

SEED = 42  # fixed random seed, so every run gives exactly the same split and results
DATA_PATH = Path("data/insurance.csv")
ARTIFACTS_DIR = Path("artifacts")  # where trained models and results are saved for app.py
ARTIFACTS_DIR.mkdir(exist_ok=True)

np.random.seed(SEED)

## 1. Load the data

One row = one insurance member. `charges` is the target (what we predict).

In [2]:
df = pd.read_csv(DATA_PATH)
print("shape:", df.shape)
df.head()

shape: (1338, 7)


,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


## 2. Cleaning

- **Duplicates**: there is 1 exact duplicate row. It's the same person counted
  twice, so we drop it.
- **Missing values**: none, so nothing to fill.
- **`sex` is dropped**: using sex to set insurance conditions is not allowed
  (EU *Test-Achats* ruling, US ACA), so the model must not use it.

In [3]:
print("duplicate rows:", df.duplicated().sum())
print("missing values:", df.isna().sum().sum())

df = df.drop_duplicates().drop(columns="sex").reset_index(drop=True)
print("shape after cleaning:", df.shape)

duplicate rows: 1
missing values: 0
shape after cleaning: (1337, 6)


## 3. Feature engineering

A linear model only understands numbers, and it can only **add up** effects.
So we:

1. **`smoker`**: text `yes`/`no` → number `1`/`0`.
2. **`region`**: 4 text values → **one-hot encoding**: one 0/1 column per
   region. We drop one region (`northeast`), which becomes the reference:
   its effect is already inside the bias `b`. Keeping all 4 would be
   redundant (they always add up to 1).
3. **`smoker_obese`** (the key feature): `1` if the member smokes **and** has
   BMI ≥ 30 (obesity), else `0`. In the data, obesity barely changes the cost
   of non-smokers, but roughly doubles the cost of smokers. A linear model
   can't discover that "both together" effect on its own, so we give it this
   column explicitly.

In [4]:
# Why smoker_obese: average cost by smoker x obese
print(
    df.assign(obese=df["bmi"] >= 30)
    .groupby(["smoker", "obese"])["charges"].mean().round(0).unstack()
)

obese     False    True 
smoker                  
no       7977.0   8856.0
yes     21363.0  41558.0


In [5]:
df["smoker"] = (df["smoker"] == "yes").astype(int)
df["smoker_obese"] = ((df["smoker"] == 1) & (df["bmi"] >= 30)).astype(int)
df = pd.get_dummies(df, columns=["region"], drop_first=True, dtype=int)

TARGET = "charges"
FEATURES = [c for c in df.columns if c != TARGET]
print("features:", FEATURES)
df.head()

features: ['age', 'bmi', 'children', 'smoker', 'smoker_obese', 'region_northwest', 'region_southeast', 'region_southwest']


,age,bmi,children,smoker,charges,smoker_obese,region_northwest,region_southeast,region_southwest
0,19,27.900,0,1,16884.92400,0,0,0,1
1,18,33.770,1,0,1725.55230,0,0,1,0
2,28,33.000,3,0,4449.46200,0,0,1,0
3,33,22.705,0,0,21984.47061,0,1,0,0
4,32,28.880,0,0,3866.85520,0,1,0,0


## 4. Split: train / validation / test (70 / 15 / 15)

- **Train**: the models learn from it.
- **Validation**: used to choose settings (e.g. the learning rate) and to
  compare feature ideas.
- **Test**: used **only once, at the very end**, to report final results on
  data no decision was based on.

**Random split** is fine: there are no dates, each row is a different person,
so there is no "future" that could leak into training. The split is
**stratified by `smoker`**, so all three sets keep the same ~20% share of
smokers (they drive most of the cost, so we don't want a test set with too
few or too many of them by chance).

In [6]:
train_df, temp_df = train_test_split(
    df, test_size=0.30, stratify=df["smoker"], random_state=SEED
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df["smoker"], random_state=SEED
)

for name, part in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"{name:5s}: {len(part):4d} rows, smokers = {part['smoker'].mean():.1%}")

train:  935 rows, smokers = 20.5%
val  :  201 rows, smokers = 20.4%
test :  201 rows, smokers = 20.4%


## 5. Standardize the features

Features have very different scales (age ~18–64, BMI ~16–53, 0/1 columns).
**Standardizing** transforms each feature to mean 0 and standard deviation 1:
`(x - mean) / std`.

- Why: gradient descent (both PyTorch versions) takes one step size (learning
  rate) for all weights. If features have very different scales, one learning
  rate is too big for some and too small for others, and training is slow or
  unstable. With the same scale, it converges well.
- We use the **same standardized inputs for all three models**, so their
  weights can be compared directly.
- The scaler is **fit on train only** (mean and std computed from train), then
  applied to val and test. Computing them on all data would let information
  from the test set leak into training.
- The target `charges` is **not** scaled: predictions stay in dollars.

In [7]:
scaler = StandardScaler()
X_train = scaler.fit_transform(train_df[FEATURES])  # learn mean/std from train only
X_val = scaler.transform(val_df[FEATURES])          # reuse train's mean/std
X_test = scaler.transform(test_df[FEATURES])

y_train = train_df[TARGET].to_numpy()
y_val = val_df[TARGET].to_numpy()
y_test = test_df[TARGET].to_numpy()

print("X_train shape:", X_train.shape)
print("train feature means (≈0):", X_train.mean(axis=0).round(2))
print("train feature stds  (≈1):", X_train.std(axis=0).round(2))

X_train shape: (935, 8)
train feature means (≈0): [ 0. -0.  0. -0.  0. -0.  0. -0.]
train feature stds  (≈1): [1. 1. 1. 1. 1. 1. 1. 1.]


## 6. Evaluation helper and naive baseline

Three metrics, all computed on predictions in dollars:

- **MAE** (main metric): average absolute error, "how many $ off per member".
- **RMSE**: like MAE but squares errors first, so big misses weigh more.
- **R²**: share of the variation in `charges` the model explains (1 = perfect,
  0 = no better than predicting the mean).

**Naive baseline**: predict the **training mean** of `charges` for everyone.
Any useful model must beat this.

In [8]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


def evaluate(y_true, y_pred):
    return {
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "R2": float(r2_score(y_true, y_pred)),
    }


baseline_value = y_train.mean()
print(f"baseline prediction for everyone: {baseline_value:,.0f} $")
print("baseline on validation:", {k: round(v, 3) for k, v in evaluate(y_val, np.full(len(y_val), baseline_value)).items()})

baseline prediction for everyone: 13,298 $
baseline on validation: {'MAE': 8556.643, 'RMSE': 11390.972, 'R2': -0.001}


## 7. Model 1: scikit-learn `LinearRegression`

Linear regression: `prediction = X @ w + b`, one weight per feature plus a
bias. scikit-learn finds the `w` and `b` that minimize the **mean squared
error (MSE)** on train directly with linear algebra (no learning rate, no
epochs). This is our reference: the two PyTorch versions should end up with
the same weights.

In [9]:
from sklearn.linear_model import LinearRegression

sk_model = LinearRegression()
sk_model.fit(X_train, y_train)

print("weights:", sk_model.coef_.round(1))
print("bias:   ", round(sk_model.intercept_, 1))
print("validation:", {k: round(v, 3) for k, v in evaluate(y_val, sk_model.predict(X_val)).items()})

weights: [3646.5  201.   637.1 5386.2 6212.1 -185.7 -306.8 -602.1]
bias:    13297.8
validation: {'MAE': 2559.835, 'RMSE': 4540.905, 'R2': 0.841}


### 7.1 Feature check (validation set)

Is `smoker_obese` worth it, and would other engineered features help? We
refit the scikit-learn model with different feature sets and compare
**validation MAE** (lower is better). The extra candidates are only built
here for the comparison:

- `smoker_bmi` = smoker × BMI (a continuous version of the interaction)
- `age_sq` = age² (cost growing faster at older ages)

In [10]:
def with_candidates(part):
    return part.assign(smoker_bmi=part["smoker"] * part["bmi"], age_sq=part["age"] ** 2)


cand_train, cand_val = with_candidates(train_df), with_candidates(val_df)
original = [f for f in FEATURES if f != "smoker_obese"]
feature_sets = {
    "A: original columns only": original,
    "B: A + smoker_obese (chosen)": FEATURES,
    "C: A + smoker_bmi": original + ["smoker_bmi"],
    "D: B + age_sq": FEATURES + ["age_sq"],
    "E: B + smoker_bmi + age_sq": FEATURES + ["smoker_bmi", "age_sq"],
}

rows = []
for name, cols in feature_sets.items():
    m = LinearRegression().fit(cand_train[cols], y_train)
    rows.append({"feature set": name, **evaluate(y_val, m.predict(cand_val[cols]))})
feature_check = pd.DataFrame(rows)
print(feature_check.round(3).to_string(index=False))

                 feature set      MAE     RMSE    R2
    A: original columns only 4036.443 5845.401 0.736
B: A + smoker_obese (chosen) 2559.835 4540.905 0.841
           C: A + smoker_bmi 2869.443 4593.073 0.837
               D: B + age_sq 2505.551 4541.792 0.841
  E: B + smoker_bmi + age_sq 2399.337 4415.047 0.850


`smoker_obese` alone cuts validation MAE by ~37% (B vs A). Adding more
features (D, E) only improves it by another 50–160 $, so we keep the simpler
set B. (No scaling is needed here: scikit-learn's exact solution gives the
same predictions with or without it.)

## 8. Model 2: PyTorch, manual training loop

Same model, but trained by **gradient descent** written by hand, exactly like
section 9.1 of the PyTorch intro notebook:

1. forward pass: `y_pred = X @ w + b`
2. loss: MSE (the same thing scikit-learn minimizes)
3. `loss.backward()`: autograd computes the gradients
4. update inside `torch.no_grad()`: `w -= lr * w.grad`, then `.grad.zero_()`

PyTorch works with **tensors**, so we convert the numpy arrays first
(`float32`, PyTorch's standard dtype).

In [11]:
import torch

torch.manual_seed(SEED)

X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32)
X_val_t = torch.tensor(X_val, dtype=torch.float32)
X_test_t = torch.tensor(X_test, dtype=torch.float32)

N_FEATURES = X_train_t.shape[1]
N_EPOCHS = 1000


def mse_loss(y_pred, y):
    return ((y_pred - y) ** 2).mean()


def train_manual(lr, n_epochs=N_EPOCHS, verbose=False):
    w = torch.zeros(N_FEATURES, requires_grad=True)
    b = torch.zeros(1, requires_grad=True)
    loss_history = []

    for epoch in range(n_epochs):
        y_pred = X_train_t @ w + b          # 1. forward pass
        loss = mse_loss(y_pred, y_train_t)  # 2. loss
        loss.backward()                     # 3. gradients

        with torch.no_grad():               # 4. update (not tracked by autograd)
            w -= lr * w.grad
            b -= lr * b.grad
        w.grad.zero_()                      # reset, or gradients accumulate
        b.grad.zero_()

        loss_history.append(loss.item())
        if verbose and epoch % 200 == 0:
            print(f"epoch {epoch:4d}  train MSE {loss.item():,.0f}")

    return w.detach(), b.detach(), loss_history

### 8.1 Choosing the learning rate (the only thing we tune)

The learning rate does not change *what* the model is (it is always the same
linear regression); it only changes *how well and how fast* gradient descent
reaches the minimum of the training loss. So we try 4 learning rates and keep
the one that reaches the **lowest training loss (MSE)** in 1000 epochs without
blowing up. Validation metrics are shown as a check.

Note the trap in the table: `lr = 0.001` gets a *lower validation MAE*, but
only by luck. It stopped before finishing training (its training loss is
clearly higher) and its validation RMSE is worse. Choosing the learning rate
by validation MAE would have picked a half-trained model.

In [12]:
lr_results = []
for lr in [0.001, 0.01, 0.1, 1.0]:
    w_try, b_try, hist = train_manual(lr)
    with torch.no_grad():
        val_pred = (X_val_t @ w_try + b_try).numpy()
    if np.isfinite(val_pred).all():
        m = evaluate(y_val, val_pred)
    else:  # training diverged (loss exploded to inf/nan)
        m = {"MAE": np.nan, "RMSE": np.nan, "R2": np.nan}
    lr_results.append({"lr": lr, "final train MSE": hist[-1], **m})

lr_table = pd.DataFrame(lr_results)
print(lr_table.to_string(index=False, float_format=lambda x: f"{x:,.3f}" if abs(x) < 10 else f"{x:,.0f}"))

BEST_LR = lr_table.loc[lr_table["final train MSE"].idxmin(), "lr"]  # idxmin skips NaN (diverged)
print("\nchosen learning rate:", BEST_LR)

   lr  final train MSE   MAE  RMSE    R2
0.001       24,541,106 2,323 4,998 0.807
0.010       20,744,896 2,559 4,541 0.841
0.100       20,744,886 2,560 4,541 0.841
1.000              NaN   NaN   NaN   NaN

chosen learning rate: 0.1


- `lr = 0.001`: too small, the loss is still going down after 1000 epochs
  (not finished).
- `lr = 1.0`: too big, each step overshoots and the loss explodes (`nan`).
- `lr = 0.01` and `0.1` both converge; `0.1` gets slightly closer to the
  minimum within the 1000 epochs, so it is chosen.

Now we train the final manual model with the chosen learning rate.

In [13]:
w_manual, b_manual, manual_loss_history = train_manual(BEST_LR, verbose=True)

print("\nweights:", w_manual.numpy().round(1))
print("bias:   ", round(b_manual.item(), 1))
with torch.no_grad():
    print("validation:", {k: round(v, 3) for k, v in evaluate(y_val, (X_val_t @ w_manual + b_manual).numpy()).items()})

epoch    0  train MSE 325,292,960
epoch  200  train MSE 20,744,886
epoch  400  train MSE 20,744,886
epoch  600  train MSE 20,744,886
epoch  800  train MSE 20,744,886

weights: [3646.5  201.   637.1 5386.2 6212.1 -185.7 -306.8 -602.1]
bias:    13297.8
validation: {'MAE': 2559.833, 'RMSE': 4540.905, 'R2': 0.841}


## 9. Model 3: PyTorch, standard workflow (`nn.Module` + `torch.optim`)

Same model again, written the way PyTorch is normally used (appendix A.1 of
the intro notebook):

- `nn.Linear(N_FEATURES, 1)` holds `w` and `b` for us.
- `torch.optim.SGD` does the update step (`optimizer.step()`) and the
  gradient reset (`optimizer.zero_grad()`).
- `nn.MSELoss()` is the same MSE.

We use **plain SGD with the same learning rate and number of epochs** as the
manual loop, so the two PyTorch versions run the exact same algorithm. The
only difference: `nn.Linear` starts from small random weights instead of
zeros, which doesn't matter once training has converged.

In [14]:
import torch.nn as nn

torch.manual_seed(SEED)
torch_model = nn.Linear(N_FEATURES, 1)
optimizer = torch.optim.SGD(torch_model.parameters(), lr=BEST_LR)
loss_fn = nn.MSELoss()

standard_loss_history = []
for epoch in range(N_EPOCHS):
    optimizer.zero_grad()                         # reset gradients
    y_pred = torch_model(X_train_t).squeeze(-1)   # forward: shape (n, 1) -> (n,)
    loss = loss_fn(y_pred, y_train_t)
    loss.backward()
    optimizer.step()                              # w -= lr * w.grad (and b)

    standard_loss_history.append(loss.item())
    if epoch % 200 == 0:
        print(f"epoch {epoch:4d}  train MSE {loss.item():,.0f}")

print("\nweights:", torch_model.weight.detach().numpy().ravel().round(1))
print("bias:   ", round(torch_model.bias.item(), 1))

epoch    0  train MSE 325,277,504
epoch  200  train MSE 20,744,886
epoch  400  train MSE 20,744,886
epoch  600  train MSE 20,744,886
epoch  800  train MSE 20,744,886

weights: [3646.5  201.   637.1 5386.2 6212.1 -185.7 -306.8 -602.1]
bias:    13297.8


## 10. Comparison: do the three agree?

All three minimize the same loss (MSE) on the same standardized training
data, so they should find (almost) the same weights. Small differences come
from PyTorch using `float32` and stopping after a fixed number of epochs,
while scikit-learn solves it exactly in `float64`.

In [15]:
def predict_all(X, X_t):
    with torch.no_grad():
        return {
            "baseline (train mean)": np.full(len(X), baseline_value),
            "scikit-learn": sk_model.predict(X),
            "PyTorch manual": (X_t @ w_manual + b_manual).numpy(),
            "PyTorch standard": torch_model(X_t).squeeze(-1).numpy(),
        }


val_preds = predict_all(X_val, X_val_t)
test_preds = predict_all(X_test, X_test_t)

rows = []
for name in val_preds:
    v, t = evaluate(y_val, val_preds[name]), evaluate(y_test, test_preds[name])
    rows.append({"model": name, "val MAE": v["MAE"], "val RMSE": v["RMSE"], "val R2": v["R2"],
                 "test MAE": t["MAE"], "test RMSE": t["RMSE"], "test R2": t["R2"]})
results_table = pd.DataFrame(rows)
print(results_table.round(3).to_string(index=False))

                model  val MAE  val RMSE  val R2  test MAE  test RMSE  test R2
baseline (train mean) 8556.643 11390.972  -0.001  9494.618  12428.481   -0.001
         scikit-learn 2559.835  4540.905   0.841  2244.107   4093.968    0.891
       PyTorch manual 2559.833  4540.905   0.841  2244.105   4093.968    0.891
     PyTorch standard 2559.833  4540.905   0.841  2244.105   4093.968    0.891


In [16]:
coef_table = pd.DataFrame({
    "scikit-learn": np.append(sk_model.coef_, sk_model.intercept_),
    "PyTorch manual": np.append(w_manual.numpy(), b_manual.item()),
    "PyTorch standard": np.append(torch_model.weight.detach().numpy().ravel(), torch_model.bias.item()),
}, index=FEATURES + ["bias"])
print(coef_table.round(2))

diff_manual = np.abs(test_preds["PyTorch manual"] - test_preds["scikit-learn"]).max()
diff_standard = np.abs(test_preds["PyTorch standard"] - test_preds["scikit-learn"]).max()
print(f"\nlargest difference vs scikit-learn on any test prediction: "
      f"manual {diff_manual:.2f} $, standard {diff_standard:.2f} $")

                  scikit-learn  PyTorch manual  PyTorch standard
age                    3646.54         3646.54           3646.54
bmi                     201.03          201.03            201.03
children                637.05          637.05            637.05
smoker                 5386.16         5386.16           5386.16
smoker_obese           6212.13         6212.13           6212.13
region_northwest       -185.70         -185.70           -185.70
region_southeast       -306.83         -306.83           -306.83
region_southwest       -602.13         -602.13           -602.13
bias                  13297.83        13297.83          13297.83

largest difference vs scikit-learn on any test prediction: manual 0.01 $, standard 0.01 $


**How to read the weights**: because features are standardized, each weight
is "how many $ the prediction changes when that feature goes up by one
standard deviation". The bias is the prediction for an average member.

## 11. Save everything the app needs

The app must **never retrain**, so we save the trained models and the data
it needs to make its plots:

- `sklearn_model.joblib`, `scaler.joblib`: scikit-learn model and the fitted
  scaler (new inputs must be scaled with the *training* mean/std).
- `manual_params.pt`: the manual model's `w` and `b`.
- `torch_model.pt`: the standard model's weights (`state_dict`).
- `dataset_with_split.csv`: the cleaned data with a `split` column, so the
  app uses exactly the same test set.
- `metrics.json`: feature list, chosen learning rate, results, loss curves.

In [17]:
import json

import joblib

joblib.dump(sk_model, ARTIFACTS_DIR / "sklearn_model.joblib")
joblib.dump(scaler, ARTIFACTS_DIR / "scaler.joblib")
torch.save({"w": w_manual, "b": b_manual}, ARTIFACTS_DIR / "manual_params.pt")
torch.save(torch_model.state_dict(), ARTIFACTS_DIR / "torch_model.pt")

split_col = pd.Series("train", index=df.index)
split_col[val_df.index] = "val"
split_col[test_df.index] = "test"
df.assign(split=split_col).to_csv(ARTIFACTS_DIR / "dataset_with_split.csv", index=False)

with open(ARTIFACTS_DIR / "metrics.json", "w") as f:
    json.dump({
        "features": FEATURES,
        "target": TARGET,
        "baseline_value": float(baseline_value),
        "best_lr": float(BEST_LR),
        "n_epochs": N_EPOCHS,
        "lr_search": lr_table.replace({np.nan: None}).to_dict(orient="records"),
        "results": results_table.to_dict(orient="records"),
        "loss_history": {"PyTorch manual": manual_loss_history, "PyTorch standard": standard_loss_history},
    }, f, indent=2)

print("saved:", sorted(p.name for p in ARTIFACTS_DIR.iterdir()))

saved: ['dataset_with_split.csv', 'manual_params.pt', 'metrics.json', 'scaler.joblib', 'sklearn_model.joblib', 'torch_model.pt']
